# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `10` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `Jenny Ruan`
- Travel and Hospitality Pipeline Lead: `Shiqi Chen`
- Retail Pipeline Lead: `Jenna Huang`
- Comparison and Visualization Lead (Integrator): `Melody Lew`

**Submission filename:** `MP03_Notebook_team_<NN>.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [ ]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

In [ ]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team 10 - melody.lew@baruchmail.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [ ]:
# Configure the Anthropic API client from Colab Secrets.
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [ ]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
#repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
!git clone https://github.com/melodylew/cis3120-spring2026.git
%cd /content/cis3120-spring2026
!git checkout mp/03-industry-comparison-team-10

repo_root = Path("/content/cis3120-spring2026")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    RETAIL_TICKERS,
    RETAIL_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")
print(f"Retail tickers: {len(RETAIL_TICKERS)}")
print(f"Retail phrases: {len(RETAIL_PHRASES)}")

---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [ ]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [ ]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [ ]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [ ]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [ ]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [ ]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers.

    Implementation hint: each EDGAR hit has hit["_source"]["tickers"];
    match case-insensitively and return only matching hits.

    Parameters
    ----------
    candidates : list[dict]
        EDGAR hits as returned by search_edgar_all_phrases.
    ticker_list : list[str]
        Tickers to retain (e.g., FINANCIAL_SERVICES_TICKERS).

    Returns
    -------
    list[dict]
        The subset of candidates whose tickers intersect ticker_list.
    """

    targets = {f"({t.upper()})" for t in ticker_list}
    filtered = []
    for hit in candidates:
        names = hit["_source"].get("display_names", [])
        names = " ".join(names).upper()
        for t in targets:
            if t in names:
                hit["ticker"] = t[1:-1]
                filtered.append(hit)
                break
    return filtered

In [ ]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> tuple[list[dict], int]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record and
    number of filtered candidates.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    ticker_list : list[str]
        Industry-specific ticker list.
    phrase_list : list[str]
        Industry-specific search-phrase list.
    window_days : int
        Length of the EDGAR date window (e.g., 30, 60, 90, 180, 360).

    Returns
    -------
    list[dict]
        One dict per geocoded location event, with the "industry" field set
        to industry_label. Records that fail classification or geocoding are
        excluded from the return value.
    int
        Number of filtered candidates.

    Implementation guidance
    -----------------------
    1. Compute start_date and end_date from window_days.
    2. Call search_edgar_all_phrases(phrase_list, start_date, end_date).
    3. Filter the candidate list with filter_candidates_by_tickers.
    4. For each filtered candidate: fetch_exhibit_text, then extract_with_claude.
    5. Keep only records where is_location_event is True.
    6. For each kept record, geocode via geocode_location; drop records that
       fail geocoding.
    7. Add the "industry" field to each surviving record.
    """

    end_date = date.today()
    start_date = end_date - timedelta(window_days)
    results = []

    candidates = search_edgar_all_phrases(phrase_list, start_date, end_date, "8-K", 2, 250)
    filtered_candidates = filter_candidates_by_tickers(candidates, ticker_list)

    for candidate in filtered_candidates:
        exhibit_text = fetch_exhibit_text(candidate)
        filing_information = extract_with_claude({"text": exhibit_text[0], "url": exhibit_text[1]})

        if not filing_information.get("is_location_event", False):
            continue

        location = geocode_location(filing_information.get("city", ""), filing_information.get("state"))
        
        if not location:
            continue

        filing_information["location"] = location
        filing_information["industry"] = industry_label

        source = candidate["_source"]
        filing_information["company"] = (source.get("display_names") or ["N/A"])[0]
        filing_information["ticker"] = candidate.get("ticker")
        filing_information["file_date"] = source.get("file_date", "N/A")
        results.append(filing_information)

    return results, len(filtered_candidates)


In [ ]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """

    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": estimated_cost_usd
    }

In [ ]:
fs_events, fs_candidate_count = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=30,
)

print(f"FS pipeline: {fs_candidate_count} candidates -> {len(fs_events)} events")

In [ ]:
th_events, th_candidates = run_industry_pipeline(
   "Travel and Hospitality",
   TRAVEL_HOSPITALITY_TICKERS,
   TRAVEL_HOSPITALITY_PHRASES,
   window_days=30,
)

print(f"Travel and Hospitality: {len(th_events)} events from {th_candidates} candidates")

In [ ]:
retail_events, retail_candidate_count = run_industry_pipeline(
    "Retail",
    RETAIL_TICKERS,
    RETAIL_PHRASES,
    window_days=30,
)

print(f"Retail pipeline: {retail_candidate_count} candidates -> {len(retail_events)} events")

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [ ]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

In [ ]:
def calculate_costs(events):
    input_tokens = sum(event.get("input_tokens", 0) for event in events)
    output_tokens = sum(event.get("output_tokens", 0) for event in events)
    return (input_tokens / 1_000_000) * 1.00 + (output_tokens / 1_000_000) * 5.00


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [ ]:
fs_events_30, fs_candidate_count_30 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=30,
)
fs_row_30 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=30,
    candidate_count=fs_candidate_count_30,
    event_count=len(fs_events_30),
    estimated_cost_usd=calculate_costs(fs_events_30),
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row_30])], ignore_index=True)

fs_events_60, fs_candidate_count_60 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=60,
)
fs_row_60 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=60,
    candidate_count=fs_candidate_count_60,
    event_count=len(fs_events_60),
    estimated_cost_usd=calculate_costs(fs_events_60),
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row_60])], ignore_index=True)

fs_events_90, fs_candidate_count_90 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=90,
)
fs_row_90 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=90,
    candidate_count=fs_candidate_count_90,
    event_count=len(fs_events_90),
    estimated_cost_usd=calculate_costs(fs_events_90),
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row_90])], ignore_index=True)

fs_events_180, fs_candidate_count_180 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=180,
)
fs_row_180 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=180,
    candidate_count=fs_candidate_count_180,
    event_count=len(fs_events_180),
    estimated_cost_usd=calculate_costs(fs_events_180),
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row_180])], ignore_index=True)

fs_events_360, fs_candidate_count_360 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=360,
)
fs_row_360 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=360,
    candidate_count=fs_candidate_count_360,
    event_count=len(fs_events_360),
    estimated_cost_usd=calculate_costs(fs_events_360),
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row_360])], ignore_index=True)

window_results

### 4.2 Window trials — Travel and Hospitality

In [ ]:
th_events_30, th_candidate_count_30 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=30,
)
th_row_30 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=30,
    candidate_count=th_candidate_count_30,
    event_count=len(th_events_30),
    estimated_cost_usd=calculate_costs(th_events_30),
)
window_results = pd.concat([window_results, pd.DataFrame([th_row_30])], ignore_index=True)

th_events_60, th_candidate_count_60 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=60,
)
th_row_60 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=60,
    candidate_count=th_candidate_count_60,
    event_count=len(th_events_60),
    estimated_cost_usd=calculate_costs(th_events_60),
)
window_results = pd.concat([window_results, pd.DataFrame([th_row_60])], ignore_index=True)

th_events_90, th_candidate_count_90 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=90,
)
th_row_90 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=90,
    candidate_count=th_candidate_count_90,
    event_count=len(th_events_90),
    estimated_cost_usd=calculate_costs(th_events_90),
)
window_results = pd.concat([window_results, pd.DataFrame([th_row_90])], ignore_index=True)

th_events_180, th_candidate_count_180 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=180,
)
th_row_180 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=180,
    candidate_count=th_candidate_count_180,
    event_count=len(th_events_180),
    estimated_cost_usd=calculate_costs(th_events_180),
)
window_results = pd.concat([window_results, pd.DataFrame([th_row_180])], ignore_index=True)

th_events_360, th_candidate_count_360 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)
th_row_360 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=360,
    candidate_count=th_candidate_count_360,
    event_count=len(th_events_360),
    estimated_cost_usd=calculate_costs(th_events_360),
)
window_results = pd.concat([window_results, pd.DataFrame([th_row_360])], ignore_index=True)

window_results

### 4.3 Window Trials - Retail

In [ ]:
retail_events_30, retail_candidate_count_30 = run_industry_pipeline(
    "Retail",
    RETAIL_TICKERS,
    RETAIL_PHRASES,
    window_days=30,
)
retail_row_30 = summarize_window_trial(
    industry_label="Retail",
    window_days=30,
    candidate_count=retail_candidate_count_30,
    event_count=len(retail_events_30),
    estimated_cost_usd=calculate_costs(retail_events_30),
)
window_results = pd.concat([window_results, pd.DataFrame([retail_row_30])], ignore_index=True)

retail_events_60, retail_candidate_count_60 = run_industry_pipeline(
    "Retail",
    RETAIL_TICKERS,
    RETAIL_PHRASES,
    window_days=60,
)
retail_row_60 = summarize_window_trial(
    industry_label="Retail",
    window_days=60,
    candidate_count=retail_candidate_count_60,
    event_count=len(retail_events_60),
    estimated_cost_usd=calculate_costs(retail_events_60),
)
window_results = pd.concat([window_results, pd.DataFrame([retail_row_60])], ignore_index=True)

retail_events_90, retail_candidate_count_90 = run_industry_pipeline(
    "Retail",
    RETAIL_TICKERS,
    RETAIL_PHRASES,
    window_days=90,
)
retail_row_90 = summarize_window_trial(
    industry_label="Retail",
    window_days=90,
    candidate_count=retail_candidate_count_90,
    event_count=len(retail_events_90),
    estimated_cost_usd=calculate_costs(retail_events_90),
)
window_results = pd.concat([window_results, pd.DataFrame([retail_row_90])], ignore_index=True)

retail_events_180, retail_candidate_count_180 = run_industry_pipeline(
    "Retail",
    RETAIL_TICKERS,
    RETAIL_PHRASES,
    window_days=180,
)
retail_row_180 = summarize_window_trial(
    industry_label="Retail",
    window_days=180,
    candidate_count=retail_candidate_count_180,
    event_count=len(retail_events_180),
    estimated_cost_usd=calculate_costs(retail_events_180),
)
window_results = pd.concat([window_results, pd.DataFrame([retail_row_180])], ignore_index=True)

retail_events_360, retail_candidate_count_360 = run_industry_pipeline(
    "Retail",
    RETAIL_TICKERS,
    RETAIL_PHRASES,
    window_days=360,
)
retail_row_360 = summarize_window_trial(
    industry_label="Retail",
    window_days=360,
    candidate_count=retail_candidate_count_360,
    event_count=len(retail_events_360),
    estimated_cost_usd=calculate_costs(retail_events_360),
)
window_results = pd.concat([window_results, pd.DataFrame([retail_row_360])], ignore_index=True)

window_results

### 4.4 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [ ]:
CHOSEN_WINDOW_DAYS = 360

fs_events, fs_candidate_count = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)
th_events, th_candidate_count = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)
retail_events, retail_candidate_count = run_industry_pipeline(
    "Retail",
    RETAIL_TICKERS,
    RETAIL_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

all_events = fs_events + th_events + retail_events
print(f"Financial Services:     {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Retail:                 {len(retail_events)} events")
print(f"Total:                  {len(all_events)} events")

---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [ ]:
m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

INDUSTRY_COLORS = {
    "Financial Services": "darkblue",
    "Travel and Hospitality": "cadetblue",
    "Retail": "purple",
}

EVENT_ICONS = {
    "opening": "home",
    "closing": "times-circle",
    "relocation": "arrows-alt",
    "expansion": "expand",
}

for event in all_events:
    # build popup HTML containing company, ticker, industry, filing date,
    # event type, summary, and a hyperlink to event["url"]
    company = event["company"]
    ticker = event["ticker"]
    industry = event["industry"]
    file_date = event["file_date"]
    event_type = event.get("event_type", "other")
    city = event["city"]
    state = event["state"]
    summary = event.get("summary", "N/A")
    url = event["url"]
    popup_html = f"""
    <b>Company:</b> {company}<br>
    <b>Ticker:</b> {ticker}<br>
    <b>Industry:</b> {industry}<br>
    <b>Filing date:</b> {file_date}<br>
    <b>Event type:</b> {event_type}<br>
    <b>Location:</b> {city}, {state}<br>
    <b>Summary:</b> {summary}<br>
    <a href="{url}">SEC filing</a>
    """

    # choose icon color from the industry label
    # choose icon name from event["event_type"]
    marker = folium.Marker(
        location=event["location"],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(
            color=INDUSTRY_COLORS.get(industry),
            icon=EVENT_ICONS.get(event_type, "info-circle"),
            prefix="fa",
        ),
    )
    marker.add_to(m)

m

### Export the map to `maps/mp03_map_team_10.html`

In [ ]:
OUTPUT_PATH = "../maps/mp03_map_team_10.html"
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")

---

## 6. Methodology

The content below also appears as a standalone Markdown file at `methodology/mp03_methodology_team_10.md`. Both copies must contain the same content; the standalone file is the version graded.

### 6.1 Ticker-list rationale

Financial Services

The seeded 14 tickers cover money-center banks, three regionals, two asset managers, two insurers, and the card networks, but Section 3 of the assignment explicitly flags the seed as biased toward retail-banking events and lacking capital-markets coverage. To close this gap, eleven tickers were added across four under-represented segments. Mid-cap regionals FITB, RF, and KEY were added because branch consolidation activity is most concentrated at this tier — the money-center banks have largely completed theirs, but mid-caps are still rationalizing post-2008 acquisitions. Capital markets names GS and MS were added because the seed had zero pure investment banks; without them the map would skew toward retail geography and miss the financial centers where IB capacity actually sits. Custody banks BK, STT, and NTRS were added because they operate the largest operations centers in the industry and are unusually frequent filers of ops-center relocation 8-Ks. Exchanges ICE, CME, and NDAQ were added because they file 8-Ks for data-center and matching-engine relocations, which represent a class of location event that no bank produces. SCHW, ALLY, and KKR were added to round out brokerage, consumer finance, and alternative asset management. No original tickers were removed.

Travel and Hospitality

The seeded 14 tickers were expanded to 75 across several segments. Casino-resort operators MGM, LVS, WYNN, CZR, PENN, BYD, and DKNG were added because they regularly file location-related 8-Ks for property openings and sportsbook launches not represented in the seed. Hotel REITs (PK, HST, APLE, RHP) were added because they file 8-Ks for individual lodging properties frequently. IHG and JBLU were added for broader hotel brand and Northeast airline coverage respectively. TRIP and ABNB were added to represent other common online travel booking agencies. Theme park operators DIS, SEAS, and EPR were added for attraction and entertainment coverage. Ground transportation tickers HTZ, CAR, UBER, and LYFT were added for rental and mobility events. I considered adding MAA but removed it as it is a residential REIT not hospitality. No original seed tickers were removed.

Retail

The retail list has around 87 publicly-traded U.S. retailers across 15+ sub-segments: big-box (WMT, TGT, COST), home improvement (HD, LOW), off-price and dollar (TJX, ROST, DG, DLTR, BURL), drug stores (WBA, CVS, RAD), specialty (BBY, GPS), grocery (KR, SFM, ACI, WMK, GO), department stores (M, KSS, JWN), beauty (ULTA, ELF, COTY, SBH), discount specialty (FIVE, OLLI, BIG, PSMT), auto (AAP, ORLY, AZO), apparel (AEO, URBN, ANF, LULU, PLCE, RL, VFC, TPR, LEVI, and others), pet retail (CHWY, PETQ, WOOF), furniture and home (W, RH, WSM, ETD, LZB), e-commerce (AMZN, EBAY, ETSY), sporting (DKS, ASO, HIBB), mattress (TPX, SNBR, PRPL), jewelry (SIG, TIF), and convenience and fuel retail (CASY, BJ, MUSA). I extended the original 14-ticker seed in three rounds. First to 20 by adding department stores, beauty, discount specialty, and auto retail, then up to around 87 to maximize event coverage at the 30-day window. Sub-segment breadth matters because drug stores trend toward net-closing while dollar stores trend toward net-opening, so a list weighted toward one sub-segment would have produced a one-sided view of expansion versus consolidation.  


### 6.2 Search-phrase rationale

Financial Services

All 10 seed phrases were retained. For corporate real estate, "headquarters relocation", "relocate its headquarters", "new headquarters", "lease termination", and "office relocation" were added to catch HQ moves and lease decisions that the retail-branch vocabulary misses. For capital markets specifically, "trading floor", "matching engine", and "exchange floor" were added because these phrases essentially only appear in IB and exchange filings, giving high precision for the segment the seed undercounted. For consolidation activity, "footprint reduction", "consolidate operations", and "office consolidation" were added to pick up cost-rationalization filings that don't actually use the word "branch" — important because some of the most relevant white-collar consolidation 8-Ks avoid retail-branch language. Finally, "expanded presence" and "new office" were added on the expansion side so the corpus would not be structurally biased toward consolidation alone. "Office" alone and "closing" alone were considered but dropped because they generate too many false positives — every governance section mentions "office", and "closing" matches every loan-closing event in bank 8-Ks.

Travel and Hospitality

All 10 seed phrases were kept and the list was expanded to 71 phrases across seven sub-segments. For hotels, "hotel acquisition" and "property acquisition" were added for REIT filings, and "all-inclusive resort", "vacation club", and "new timeshare" were added for VAC and HGV coverage. For airlines, "new concourse", "new crew base", and "new training center" were added for facility events beyond route launches. For cruise, "private island" and "new itinerary" were added as well as "private island" where there is high precision, appearing almost exclusively in CCL and RCL destination opening filings. For gaming, "new sportsbook" and "retail sportsbook" were added for DKNG and PENN physical location openings. For theme parks, "new ride" and "new land" were added as it shows higher accuracy in Disney themed land announcements. I changed "homeport" to "home port" as cruise lines commonly write it as one word in actual filings.

Retail

The retail phrase list has around 76 phrases across three categories. For expansion: "new store," "store opening," "grand opening," "flagship store," "new location," "new format," "opened a new," "announced the opening," "plans to open," "first store," "expanding," "expansion," "new market," "market entry," "international expansion," and others. For consolidation: "store closure," "store closing," "store closures," "store consolidation," "plans to close," "to close," "will close," "announced the closure," "closing stores," "closure of," "footprint reduction," and others. For supply-chain and real estate: "distribution center," "fulfillment center," "warehouse opening," "distribution facility," "store relocation," "omnichannel hub," "facility opening," "facility closure," "sale-leaseback," "property acquisition," "lease termination," "real estate," and others. I extended the original 10-phrase seed to 16 in round 2 to cover supply-chain language, then to around 76 in round 3 to include the corporate 8-K language that smaller phrase lists miss. The broader phrases trade precision for recall, which matches the Module 15 pipeline design where Stage 1 retrieves widely and Stage 3 filters strictly.


### 6.3 Window-experiment results

| | industry | window_days | candidate_count | event_count | estimated_cost_usd |
|---|---|---|---|---|---|
| 0 | Financial Services | 30 | 21 | 3 | 0.008127 |
| 1 | Financial Services | 60 | 24 | 5 | 0.013549 |
| 2 | Financial Services | 90 | 25 | 5 | 0.013574 |
| 3 | Financial Services | 180 | 48 | 7 | 0.019493 |
| 4 | Financial Services | 360 | 88 | 9 | 0.024986 |
| 5 | Travel and Hospitality | 30 | 17 | 3 | 0.008320 |
| 6 | Travel and Hospitality | 60 | 21 | 3 | 0.008320 |
| 7 | Travel and Hospitality | 90 | 29 | 5 | 0.014298 |
| 8 | Travel and Hospitality | 180 | 9 | 3 | 0.008780 |
| 9 | Travel and Hospitality | 360 | 17 | 5 | 0.015829 |
| 10 | Retail | 30 | 20 | 1 | 0.002636 |
| 11 | Retail | 60 | 29 | 3 | 0.006982 |
| 12 | Retail | 90 | 55 | 4 | 0.009874 |
| 13 | Retail | 180 | 80 | 2 | 0.004674 |
| 14 | Retail | 360 | 92 | 2 | 0.005395 |

360 is the appropriate window because the Travel and Hospitality and Retail industries do not get the required 8 events. Additionally, the total cost for all trials is ~$0.16 which is under the $3 limit.

### 6.4 Stage 3 classification quality per industry

Financial Services

At the 30-day window, Stage 3 processed 2 candidates and classified 0 as location events. Both were quarterly earnings releases that mentioned branch-related language in boilerplate sections, not actual location announcements. At the 360-day window, the pipeline identified multiple true location events including branch openings by National Bankshares (Roanoke, VA), Unity Bancorp (Madison, NJ), Farmers & Merchants Bancorp (Livingston, CA), First United (Boonsboro, MD), and Amerant Bank (Miami Beach, FL), plus one closing by First Northwest Bank (Bellevue, WA). The main false positive pattern was investor presentations referencing branch counts statistically rather than announcing a specific new event.


Travel and Hospitality 

Stage 3 has classified 7 of 17 filtered candidates as location events at the 360-day window, yielding 4 openings and 3 expansions with clean, complete field extraction. The 10 false positives were relatively precise in correctly filtering out 4 Caesars quarterly earnings releases, 3 credit/debt facility amendments (Travel & Leisure, Park Hotels), and a Hyatt Hotels Corp agreement as generic corporate disclosures. However, potential false negatives remain unverified for a Six Flags news release and a DiamondRock exhibit. Two key error patterns emerged: two different phrases must have matched filings that describe the same event in the candidates list which got classified twice by Stage 3, leading to dual records for both the Hollywood Casino Joliet and Royal Palm South Beach Miami events. A misclassification occurred on a Phoenix record, where an acquisition (change of ownership) was flagged as a new facility "opening".

Retail

At the 30-day window, Stage 3 processed 19 candidates and classified 1 as a confirmed location event. With only one event, a true precision rate isn't statistically meaningful, but the 19-to-1 funnel itself is informative. It reflects the recall-over-precision choice in 6.2, where the 76-phrase seed casts wide on purpose. The filtered candidates appeared to fall into three patterns: earnings releases using "expansion" in a financial sense, routine business overviews mentioning past store counts, and lease or real-estate disclosures that weren't tied to a specific new opening or closing. The most plausible false-negative risk is small-format moves that retailers disclose informally rather than in an 8-K, which the pipeline can't recover. The low conversion is also consistent with retail filing behavior at a short window. Most 30-day 8-K activity is earnings, executive changes, and dividends, not store-level real estate, so a longer window is likely necessary for retail to produce a comparable event count. 


### 6.5 Limitations

Financial Services 
The primary limitation was phrase mismatch — terms like "data center" and "executive offices" generated high false positive volumes from earnings releases rather than location announcements. A second limitation was that large banks like JPMorgan rarely file dedicated 8-Ks for individual branch decisions, so the pipeline undercounts events for the largest institutions while overcounting false positives from smaller community banks whose investor presentations happen to contain branch-related language.
Travel and Hospitality  

The pipeline’s primary limitation was that it failed to find the target of 8 events even after searching a full year of data, because travel and hospitality companies usually announce new locations in press releases or investor meetings rather than official SEC 8-K filings. Second, because the system looks for each search phrase separately and doesn't remove duplicate results, the exact same real-world events (like the Joliet and Miami locations) were counted and mapped multiple times, making the map look more crowded than it actually is. Third of all, searching for a longer timeframe didn’t always bring back more data; for example, the 180-day window trial brought back fewer results than the 90-day trial because of system lag and temporary website errors on the SEC database.

Retail

Four limitations shape how retail can be read in the comparative reflection. First, sub-segment selection is itself an analytical choice. The 87-ticker list leans toward big-box, dollar, drug, off-price, and supply-chain real estate, and a list weighted toward apparel or grocery would produce a different map. The retail pattern should be read as one defensible slice of public retail, not the full industry. Second, the 8-K disclosure threshold makes small and independent retailers invisible, which undercuts coverage of a meaningful share of U.S. store-level activity. Third, the recall-over-precision phrase choice lowers expected Stage 3 precision relative to a tighter list. Verifying it tightly would require manual review of the full candidate set. Fourth, the 30-day window produced only one confirmed retail event, which limits how much the reflection can claim about retail's geography at this window. Longer windows are likely needed for retail to support meaningful comparison with Financial Services and Travel & Hospitality. 

---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_<NN>.md`.

*TODO: Write the comparative reflection here. Mere description of the maps does not earn full credit; the reflection must offer substantive interpretation grounded in the underlying business economics and address limitations honestly.*

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.